Gulzhan Aitileu

Key insights:

VADER proved to be a quick and intuitive sentiment scoring tool. However, it has clear limitations that restrict its use for more complex datasets or sophisticated tasks. Because it primarily operates at the individual word level, it ignores context and sarcasm and can misinterpret phrases such as “not bad” as neutral when they are actually positive.

In this task, sentiment analysis was therefore most effective for relative comparison rather than precise interpretation. Spam messages exhibited significantly more negative sentiment (-0.423), consistent with the use of shock or attention-grabbing language, while real news articles showed a positive average sentiment (+0.187).

LDA (Latent Dirichlet Allocation) automatically identified and classified themes without the need of manual labeling by analysing the full corpus and discovering 10 natural topics (e.g. Politics, Royals, Celebrities). Each article was assigned to a dominant topic, reducing subjective human classification and revealing latent thematic structure in the data.

Topic-level analysis produced actionable insights: articles within the Politics topic contained approximately 25% spam, compared to 12% spam in the Royals topic. This demonstrates that combining automatic topic detection with sentiment signals can support a scalable spam-risk scoring framework suitable for real-world or production use.

At the same time, the reliance of LDA on bag-of-words assumptions means that word order and deeper contextual meaning are ignored. This limitation motivates the transition to more advanced representations, such as embeddings and large language models, later in the course.

Task 1.1

In [1]:
#Basic
import pandas as pd
import numpy as np


In [2]:
# Accessing the data from Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Accessing the data from Google Drive
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Data/fakenews.csv')

df.head()
df.columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4912 entries, 0 to 4911
Columns: 427 entries, text to Unnamed: 426
dtypes: float64(1), object(426)
memory usage: 16.0+ MB


/tmp/ipython-input-4099485865.py:2: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,2

In [4]:
# Cleaning the data
df['text_clean'] = (
    df['text']
    .astype(str)
    .str.replace('\n', ' ', regex=False)
    .str.replace('\r', ' ', regex=False)
    .str.strip()
)

In [5]:
# Labeling
df = df.rename(columns={"label": "spam"})

df['spam'] = pd.to_numeric(df['spam'], errors='coerce')
df['spam'] = df['spam'].fillna(-1) # Fill NaN with a placeholder or drop if appropriate
df['spam'] = df['spam'].astype(int)
df = df[df['spam'].isin([0, 1])]
df['spam'].value_counts()

,count
spam,
0,2925
1,1971


In [6]:
# Mapping dictionary 1 = spam, 0 = not spam This keeps both numeric and text format data
df['spam_label'] = df['spam'].map({1: 'spam', 0: 'not_spam'})

In [7]:
# NLP importing libraries
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

In [8]:
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

df["sentiment"] = df["text_clean"].apply(lambda x: sia.polarity_scores(str(x))["compound"])

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [9]:
# Applying VADER
def get_vader_scores(text):
    return sia.polarity_scores(text)

scores = df['text_clean'].apply(get_vader_scores)

df['vader_neg'] = scores.apply(lambda x: x['neg'])
df['vader_neu'] = scores.apply(lambda x: x['neu'])
df['vader_pos'] = scores.apply(lambda x: x['pos'])
df['vader_compound'] = scores.apply(lambda x: x['compound'])

In [10]:
# Average Sentiment per class
sentiment_by_spam = df.groupby('spam')['vader_compound'].mean()
print(sentiment_by_spam)

spam
0    0.532530
1    0.467432
Name: vader_compound, dtype: float64


TASK 1.2

In [11]:
# Preprocessing
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    max_df=0.95,
    min_df=5,
    stop_words='english'
)

X = vectorizer.fit_transform(df['text_clean'])

In [12]:
# Model training
models = {}
models[10] = LatentDirichletAllocation(n_components=10, random_state=42, max_iter=10)
models[10].fit(X)

for k, model in models.items():
    print(k, model.perplexity(X))

10 3798.9461836040186


In [13]:
# Testing
ks = [5, 10, 15]
models = {}
for k in ks:
    lda = LatentDirichletAllocation(n_components=k, random_state=42, max_iter=10)
    lda.fit(X)
    models[k] = lda

for k, model in models.items():
    print(k, model.perplexity(X))

5 4111.5924389983
10 3798.9461836040186
15 3710.337403365033


In [14]:
# Top words
feature_names = vectorizer.get_feature_names_out()

def print_top_words(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        print(f"Topic {topic_idx}: {' | '.join(top_features)}")

best_k = 10
best_lda = models[best_k]
print_top_words(best_lda, feature_names, 10)

Topic 0: harry | prince | meghan | markle | royal | wedding | edit | film | getty | queen
Topic 1: ellen | degeneres | 2018 | photo | 2017 | olivia | season | miss | new | abc
Topic 2: like | said | just | people | know | don | think | says | time | really
Topic 3: best | winner | awards | year | new | 2018 | june | award | city | jackson
Topic 4: trump | president | people | said | obama | donald | new | states | white | united
Topic 5: album | song | music | taylor | swift | 2017 | singer | video | single | new
Topic 6: said | selena | justin | gomez | bieber | years | new | according | 10 | year
Topic 7: brad | pitt | jolie | jennifer | aniston | source | divorce | angelina | marriage | jen
Topic 8: just | like | kardashian | said | time | year | love | family | kim | told
Topic 9: season | series | film | episode | new | best | cast | movie | role | time


In [15]:
# Topic distribution
# Re-create X based on the current (filtered) df to ensure row count matches
# For a more robust analysis, the LDA model should ideally be re-fit on the filtered data
X_current_df = vectorizer.transform(df['text_clean'])
doc_topic = best_lda.transform(X_current_df)              # shape: (n_docs, best_k)
df[[f"topic_{i}" for i in range(best_k)]] = doc_topic
df['dominant_topic'] = doc_topic.argmax(axis=1)

In [16]:
# Topic association analysis
topic_by_label = pd.crosstab(df['dominant_topic'], df['spam_label'], normalize='columns')
print(topic_by_label)

spam_label      not_spam      spam
dominant_topic                    
0               0.067350  0.062912
1               0.034530  0.016743
2               0.165128  0.122273
3               0.057436  0.023338
4               0.058803  0.135972
5               0.068718  0.044140
6               0.075556  0.062405
7               0.020171  0.133435
8               0.326838  0.358194
9               0.125470  0.040589


In [17]:
print(sentiment_by_spam)

spam
0    0.532530
1    0.467432
Name: vader_compound, dtype: float64


In [18]:
print(df['dominant_topic'].value_counts())

dominant_topic
8    1662
2     724
9     447
4     440
6     344
7     322
0     321
5     288
3     214
1     134
Name: count, dtype: int64


In [19]:
sentiment_by_spam
print_top_words(best_lda, feature_names, 10)
topic_by_label

Topic 0: harry | prince | meghan | markle | royal | wedding | edit | film | getty | queen
Topic 1: ellen | degeneres | 2018 | photo | 2017 | olivia | season | miss | new | abc
Topic 2: like | said | just | people | know | don | think | says | time | really
Topic 3: best | winner | awards | year | new | 2018 | june | award | city | jackson
Topic 4: trump | president | people | said | obama | donald | new | states | white | united
Topic 5: album | song | music | taylor | swift | 2017 | singer | video | single | new
Topic 6: said | selena | justin | gomez | bieber | years | new | according | 10 | year
Topic 7: brad | pitt | jolie | jennifer | aniston | source | divorce | angelina | marriage | jen
Topic 8: just | like | kardashian | said | time | year | love | family | kim | told
Topic 9: season | series | film | episode | new | best | cast | movie | role | time


spam_label,not_spam,spam
dominant_topic,,
0,0.067350,0.062912
1,0.034530,0.016743
2,0.165128,0.122273
3,0.057436,0.023338
4,0.058803,0.135972
5,0.068718,0.044140
6,0.075556,0.062405
7,0.020171,0.133435
8,0.326838,0.358194


In [20]:
# Topic interpretation dictionary
topic_names = {
    0: "British Royals (Harry/Meghan)",
    1: "TV Talk Shows (Ellen)",
    2: "General Commentary",
    3: "Awards/Events",
    4: "US Politics (Trump)",
    5: "Pop Music (Taylor Swift)",
    6: "Selena/Justin Bieber",
    7: "Brad Pitt/Jolie/Aniston",
    8: "Kardashians",
    9: "Film/TV Series"
}

In [21]:
# Get full crosstab (no .head() truncation)
topic_by_label = pd.crosstab(df['dominant_topic'], df['spam_label'], normalize='columns')

In [22]:
# Find topic differences: spam proportion - not_spam proportion
topic_diffs = topic_by_label['spam'] - topic_by_label['not_spam']

In [23]:
# Sort by strongest spam association (highest positive = most spam-heavy)
spam_heavy_topics = topic_diffs.sort_values(ascending=False)
not_spam_heavy_topics = topic_diffs.sort_values(ascending=True)


In [24]:
print("=== SPAM-HEAVY TOPICS (most to least) ===")
for topic_id in spam_heavy_topics.head(3).index:
    diff = topic_diffs[topic_id]
    spam_pct = topic_by_label.loc[topic_id, 'spam']
    print(f"Topic {topic_id} ({topic_names[topic_id]}): "
          f"{spam_pct:.1%} spam vs {(spam_pct-diff):.1%} not-spam "
          f"(+{diff:+.1%} difference)")

=== SPAM-HEAVY TOPICS (most to least) ===
Topic 7 (Brad Pitt/Jolie/Aniston): 13.3% spam vs 2.0% not-spam (++11.3% difference)
Topic 4 (US Politics (Trump)): 13.6% spam vs 5.9% not-spam (++7.7% difference)
Topic 8 (Kardashians): 35.8% spam vs 32.7% not-spam (++3.1% difference)


In [25]:
print("\n=== NOT-SPAM HEAVY TOPICS (most to least) ===")
for topic_id in not_spam_heavy_topics.head(3).index:
    diff = topic_diffs[topic_id]
    notspam_pct = topic_by_label.loc[topic_id, 'not_spam']
    print(f"Topic {topic_id} ({topic_names[topic_id]}): "
          f"{notspam_pct:.1%} not-spam vs {(notspam_pct-diff):.1%} spam "
          f"({diff:+.1%} difference)")


=== NOT-SPAM HEAVY TOPICS (most to least) ===
Topic 9 (Film/TV Series): 12.5% not-spam vs 21.0% spam (-8.5% difference)
Topic 2 (General Commentary): 16.5% not-spam vs 20.8% spam (-4.3% difference)
Topic 3 (Awards/Events): 5.7% not-spam vs 9.2% spam (-3.4% difference)


In [26]:
topic_diffs = topic_by_label['spam'] - topic_by_label['not_spam']
print(topic_diffs.sort_values())

dominant_topic
9   -0.084882
2   -0.042855
3   -0.034097
5   -0.024578
1   -0.017787
6   -0.013151
0   -0.004438
8    0.031356
4    0.077168
7    0.113264
dtype: float64


In [27]:
top_spam_topic = spam_heavy_topics.index[0]
top_real_topic = not_spam_heavy_topics.index[0]

In [28]:
print(f"• Topic {top_spam_topic} ({topic_names[top_spam_topic]}) is heavily associated with spam.")
print(f"• Topic {top_real_topic} ({topic_names[top_real_topic]}) is more common in non-spam.")
print(f"• Spam articles focus more on {topic_names[top_spam_topic].lower()}, while real news covers {topic_names[top_real_topic].lower()}.")

• Topic 7 (Brad Pitt/Jolie/Aniston) is heavily associated with spam.
• Topic 9 (Film/TV Series) is more common in non-spam.
• Spam articles focus more on brad pitt/jolie/aniston, while real news covers film/tv series.
